# H1 · Classical multi-mission planning

Use **Lunar H0 (Python 3.12)** and run top to bottom. This is the existing local kernel;
no new dependency, network call, quantum job or credential is required.

We compare independent choices with a bounded exact joint search, using the same H0
inputs and an independent constraint validator. All supplied mission data are synthetic.
A feasible result satisfies the declared static scheduling rules, **not landing safety**.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from html import escape
import os
import sys
from IPython.display import HTML, display

start = Path(os.environ.get("LUNAR_REPO_ROOT", Path.cwd())).resolve()
candidates = [start, *start.parents, start / "h0-hybrid-workspace"]
ROOT = next((p for p in candidates if (p / "backend/app/hybrid_planning.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open from the repository or set LUNAR_REPO_ROOT before starting the kernel.")
sys.path.insert(0, str(ROOT))
from shared.hybrid import Scenario, load_scenario, fingerprint
from backend.app.hybrid_planning import (
    CandidateInput, SolverResult, generate_candidates,
    select_independent, solve_exact, validate_plan,
)
import backend.app.hybrid_planning as planning_module
assert Path(planning_module.__file__).resolve() == ROOT / "backend/app/hybrid_planning.py"
print("Python:", sys.version.split()[0])
print("Repository:", ROOT)
print("H1: classical execution only; quantum jobs submitted: 0")

def show_table(headers, rows):
    def row(values, tag):
        return "<tr>" + "".join(f"<{tag}>{escape(str(v))}</{tag}>" for v in values) + "</tr>"
    display(HTML("<table>" + row(headers, "th") + "".join(row(r, "td") for r in rows) + "</table>"))

Python: 3.12.4
Repository: /Users/tk/.codex/worktrees/task1-ingestion-cache/Nasa-Space-Apps-Challenge
H1: classical execution only; quantum jobs submitted: 0


## 1. Load the original H0 problem

Three missions have four options each: **64 assignments** before constraints.
The common antenna has one channel. Option scores are 10, 9, 7 and 6; their objective
scale is 10. Times are UTC and intervals are half-open: adjacent use does not overlap.

In [2]:
examples = ROOT / "shared/examples/hybrid"
scenarios = {p.stem: load_scenario(p) for p in sorted(examples.glob("*.json"))}
scenario = scenarios["shared-conflict"]
show_table(["Option", "Mission", "Start UTC", "End UTC", "Synthetic science score"], [
    [c.option_id, c.mission_id, c.window.start_utc.isoformat(), c.window.end_utc.isoformat(), c.metrics["science_value"]]
    for c in scenario.candidates
])
print("Input fingerprint:", fingerprint(scenario))

Option,Mission,Start UTC,End UTC,Synthetic science score
mission-1-option-1,mission-1,2026-11-04T00:00:00+00:00,2026-11-04T01:00:00+00:00,10.0
mission-1-option-2,mission-1,2026-11-04T01:00:00+00:00,2026-11-04T02:00:00+00:00,9.0
mission-1-option-3,mission-1,2026-11-04T02:00:00+00:00,2026-11-04T03:00:00+00:00,7.0
mission-1-option-4,mission-1,2026-11-04T03:00:00+00:00,2026-11-04T04:00:00+00:00,6.0
mission-2-option-1,mission-2,2026-11-04T00:00:00+00:00,2026-11-04T01:00:00+00:00,10.0
mission-2-option-2,mission-2,2026-11-04T01:00:00+00:00,2026-11-04T02:00:00+00:00,9.0
mission-2-option-3,mission-2,2026-11-04T02:00:00+00:00,2026-11-04T03:00:00+00:00,7.0
mission-2-option-4,mission-2,2026-11-04T03:00:00+00:00,2026-11-04T04:00:00+00:00,6.0
mission-3-option-1,mission-3,2026-11-04T00:00:00+00:00,2026-11-04T01:00:00+00:00,10.0
mission-3-option-2,mission-3,2026-11-04T01:00:00+00:00,2026-11-04T02:00:00+00:00,9.0


Input fingerprint: dd410a8ff30a5e16d701a87f19a6c3f5d3ff5597efc43821606615c3e2f0284d


## 2. Prepare candidates from explicit site/time records

The preparation function assigns content-based IDs to supplied alternatives. It does not
invent site/time-dependent scores, visibility, fuel, communication windows or energy demand.
Order is stable; duplicates are rejected. Changing content changes its ID. Rebuild explicit
ID-based incompatibilities when generating new IDs. H3/H4 connect validated scientific models.

In [3]:
inputs = tuple(CandidateInput.model_validate(c.model_dump(exclude={"option_id"})) for c in scenario.candidates)
generated = generate_candidates(inputs)
assert generated == generate_candidates(tuple(reversed(inputs)))
assert len(generated) == 12
print("Prepared candidates:", len(generated))
print("Example generated ID:", generated[0].option_id)
print("Sources preserved:", generated[0].source_ids)
print("The following comparison keeps the original readable H0 option IDs.")

Prepared candidates: 12
Example generated ID: option-10b112780fc3ebc8769ccfff593429e70848c66c38b23600c96a9273e4c3f910
Sources preserved: ('teaching-inputs-v1',)
The following comparison keeps the original readable H0 option IDs.


## 3. Independent choices: high score, conflicting resource demand

Each mission takes its own best score, ignoring joint constraints during selection.
The validator then checks the complete proposal. A score can be calculated for an invalid
plan; it must not be displayed as an achievable result.

In [4]:
independent = select_independent(scenario)
assert independent.status == "proposal"
assert independent.validation.status == "infeasible"
assert independent.validation.exact_score == "3"
print("Selected:", independent.plan.selected_option_ids)
print("Raw science score:", independent.validation.metric_totals["science_value"])
print("Normalized score:", independent.validation.score)
print("Validation:", independent.validation.status)
show_table(["Rule", "Resource", "Required", "Capacity", "Unit", "Explanation"], [
    [i.code, i.resource_id, i.required, i.capacity, i.unit, i.message]
    for i in independent.validation.issues
])

Selected: ('mission-1-option-1', 'mission-2-option-1', 'mission-3-option-1')
Raw science score: 30.0
Normalized score: 3.0
Validation: infeasible


Rule,Resource,Required,Capacity,Unit,Explanation
capacity,shared-antenna,3,1,channel,Simultaneous demand exceeds available channel capacity.


## 4. Exact joint planning

Enumerate every assignment and check the original constraints. Maximize the weighted,
normalized score only among feasible plans. Exact decimal-rational comparisons avoid
accidental capacity failures such as 0.1 + 0.2 exceeding 0.3. Display scores remain floats.

In [5]:
joint = solve_exact(scenario)
assert joint.status == "optimal"
assert joint.evaluated_combinations == 64
assert joint.feasible_count == 24
assert joint.validation.exact_score == "13/5"
assert joint.optimal_plan_count == 6
assert validate_plan(joint.plan, scenario) == joint.validation
show_table(["Method", "Validation", "Raw science", "Normalized score", "Evaluated", "Elapsed s"], [
    [r.solver_name, r.validation.status, r.validation.metric_totals["science_value"], r.validation.score,
     r.evaluated_combinations, round(r.elapsed_s, 6)] for r in (independent, joint)
])
print("Feasible assignments:", joint.feasible_count)
print("Infeasible assignments:", joint.infeasible_count)
print("Equally optimal plans:", joint.optimal_plan_count)
print("Chosen:", joint.plan.selected_option_ids)
print("Reject reasons (counts of plans):", joint.failure_counts)
print("Tie-break: lexicographically smallest sorted option IDs.")

Method,Validation,Raw science,Normalized score,Evaluated,Elapsed s
h1-independent-v1,infeasible,30.0,3.0,1,0.003358
h1-exact-v1,feasible,26.0,2.6,64,0.011245


Feasible assignments: 24
Infeasible assignments: 40
Equally optimal plans: 6
Chosen: ('mission-1-option-1', 'mission-2-option-2', 'mission-3-option-3')
Reject reasons (counts of plans): {'capacity': 40}
Tie-break: lexicographically smallest sorted option IDs.


## 5. Shared timeline

Bars show selected candidate intervals, not flight trajectories. In this teaching case,
antenna demand covers each entire candidate interval. Red bars show the invalid
independent proposal; green bars show the jointly feasible plan. Hover over a bar for its ID.

In [6]:
# Presentation only: draw intervals already selected by the backend.
svg = ['<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 900 350" role="img" aria-label="Independent and joint mission timelines">',
       '<rect width="900" height="350" fill="#0e1726"/>']
epoch = scenario.window.start_utc
left, hour_width = 210, 145
for hour in range(5):
    x = left + hour * hour_width
    svg.append(f'<path d="M{x} 40 V320" stroke="#334155"/><text x="{x}" y="27" fill="#d1d5db" font-size="13">{hour:02d}:00</text>')
lookup = {c.option_id: c for c in scenario.candidates}
for group_index, (label, result, color) in enumerate((
    ("Independent", independent, "#fb7185"), ("Joint", joint, "#34d399")
)):
    for index, option_id in enumerate(result.plan.selected_option_ids):
        candidate = lookup[option_id]
        start_h = (candidate.window.start_utc - epoch).total_seconds() / 3600
        width_h = (candidate.window.end_utc - candidate.window.start_utc).total_seconds() / 3600
        y = 55 + group_index * 145 + index * 38
        svg.append(f'<text x="12" y="{y+20}" fill="#e5e7eb" font-size="13">{escape(label)} · {escape(candidate.mission_id)}</text>')
        svg.append(f'<rect x="{left + start_h*hour_width}" y="{y}" width="{width_h*hour_width}" height="27" rx="4" fill="{color}"><title>{escape(option_id)}</title></rect>')
svg.append('</svg>')
display(HTML(''.join(svg)))

## 6. Change capacity explicitly

Modify a copy, increase the scenario revision, and solve again. These are declared
resource alternatives, not telemetry updates. Capacity zero is known unavailability;
capacity `None` is unknown. Each scenario gets a different input fingerprint.

In [7]:
capacity_results = []
expected = {0: ("infeasible", None), 1: ("optimal", "13/5"), 2: ("optimal", "29/10"), 3: ("optimal", "3")}
for capacity in range(4):
    raw = scenario.model_dump(mode="json")
    raw["revision"] += 1
    raw["resources"][0]["capacity"] = capacity
    variant = Scenario.model_validate(raw)
    result = solve_exact(variant)
    score = result.validation.exact_score if result.validation else None
    assert (result.status, score) == expected[capacity]
    capacity_results.append([capacity, result.status, result.feasible_count,
                             result.validation.score if result.validation else None])
show_table(["Channels", "Status", "Feasible assignments", "Best normalized score"], capacity_results)

Channels,Status,Feasible assignments,Best normalized score
0,infeasible,0,None
1,optimal,24,2.6
2,optimal,60,2.9
3,optimal,64,3.0


## 7. Infeasible, incomplete and unsupported are different

Missing any candidate objective or a positively demanded resource capacity blocks the
solver conservatively; it does not silently discard unknown alternatives. A validator
can still check a complete individual plan that does not use an unknown alternative.

All event declarations are unsupported in H1, including outages. A caller must explicitly
prepare a static resource/state snapshot before using H1; automatic replay/replanning is later work.

In [8]:
benchmark_results = {name: solve_exact(value) for name, value in scenarios.items()}
assert benchmark_results["infeasible"].status == "infeasible"
assert benchmark_results["missing-capacity"].status == "incomplete"
assert benchmark_results["resource-outage"].status == "unsupported"
show_table(["Scenario", "Status", "Evaluated", "Feasible", "Explanation"], [
    [name, result.status, result.evaluated_combinations, result.feasible_count,
     "; ".join(i.message for i in result.issues)] for name, result in benchmark_results.items()
])

Scenario,Status,Evaluated,Feasible,Explanation
independent,optimal,64,64,
infeasible,infeasible,64,0,
missing-capacity,incomplete,0,0,Required resource capacity is unknown.
resource-outage,unsupported,0,0,H1 accepts static inputs only; event declarations require explicit state/resource updates before planning.
shared-conflict,optimal,64,24,


## 8. A bounded search is not an optimality certificate

Stop after seven evaluations. An incumbent is available in this ordering, but there are
unexamined assignments. Even if its score happens to equal the full-search answer,
the bounded run itself has not proven optimality. The limit counts assignments, not seconds.

In [9]:
limited = solve_exact(scenario, max_evaluations=7)
assert limited.status == "limit_reached" and not limited.search_complete
assert limited.plan is not None and limited.optimal_plan_count is None
print("Status:", limited.status)
print("Evaluated:", limited.evaluated_combinations, "/", limited.total_combinations)
print("Best found score:", limited.validation.score)
print(limited.issues[0].message)

Status: limit_reached
Evaluated: 7 / 64
Best found score: 2.6
Examined 7 of 64 assignments; optimum is not certified.


## 9. Cumulative energy budget

Add an explicitly synthetic **10 J per mission** consumption and compare 25 J versus
30 J total capacity. This is a cumulative budget, not a battery/recharge/illumination model.
It is counted across the entire scenario, even when missions occur at different times.

In [10]:
energy_rows = []
for capacity_j in (25, 30):
    raw = scenario.model_dump(mode="json")
    raw["revision"] += 1
    raw["sources"].append({"source_id": "h1-energy-assumption", "kind": "assumption",
                           "description": "Teaching example only: 10 joules per selected mission; no recharge."})
    raw["resources"].append({"resource_id": "energy-demo", "mode": "budget", "unit": "J",
                             "capacity": capacity_j, "availability": [raw["window"]],
                             "source_ids": ["h1-energy-assumption"]})
    for candidate in raw["candidates"]:
        candidate["demands"].append({"resource_id": "energy-demo", "amount": 10,
                                     "unit": "J", "window": candidate["window"]})
        candidate["source_ids"].append("h1-energy-assumption")
    energy_result = solve_exact(Scenario.model_validate(raw))
    assert energy_result.status == ("infeasible" if capacity_j == 25 else "optimal")
    energy_rows.append([capacity_j, energy_result.status, energy_result.failure_counts])
show_table(["Budget J", "Status", "Rejected-plan counts by rule"], energy_rows)

Budget J,Status,Rejected-plan counts by rule
25,infeasible,"{'budget': 64, 'capacity': 40}"
30,optimal,{'capacity': 40}


## 10. Export evidence and reject stale results

A result includes the full input fingerprint, selected plan, validation, counts, elapsed
seconds and solver identity. H0 Plan remains `unvalidated`; H1 evidence is the separate
ValidationReport. Revalidate an imported result against its original input before using it.

In [11]:
with TemporaryDirectory() as folder:
    path = Path(folder) / "h1-result.json"
    path.write_text(joint.model_dump_json(indent=2), encoding="utf-8")
    loaded = SolverResult.model_validate_json(path.read_text(encoding="utf-8"))
    assert validate_plan(loaded.plan, scenario) == loaded.validation
print("JSON export/reload and independent validation passed; temporary export removed.")
changed_raw = scenario.model_dump(mode="json")
changed_raw["resources"][0]["capacity"] = 2
changed = Scenario.model_validate(changed_raw)
try:
    validate_plan(joint.plan, changed)
except ValueError as error:
    print("Expected stale-input rejection:", error)
else:
    raise AssertionError("Stale plan was accepted")

JSON export/reload and independent validation passed; temporary export removed.
Expected stale-input rejection: Plan input identity does not match scenario


## 11. Reproducibility

Selection, scores, counts and tie-breaking are deterministic. Runtime is measured rather
than deterministic and is excluded from this comparison. The current exact problem is
small enough for classical enumeration; no quantum advantage has been demonstrated.

In [12]:
repeated = solve_exact(scenario)
assert repeated.model_dump(exclude={"elapsed_s"}) == joint.model_dump(exclude={"elapsed_s"})
print("Same input -> same selected plan, score, feasibility counts and tie count.")
print("H1 complete. Next: map the same problem to QUBO/QAOA, then reuse validate_plan.")
print("No quantum jobs were submitted by this notebook.")

Same input -> same selected plan, score, feasibility counts and tie count.
H1 complete. Next: map the same problem to QUBO/QAOA, then reuse validate_plan.
No quantum jobs were submitted by this notebook.


## Scope and next experiments

H1 validates static option selection, deadlines, declared availability, concurrent channels,
cumulative joules and explicit exclusions. It does not propagate trajectories, evaluate
radio links, replay events, model recharge or generate science/fuel metrics from physics.

H2 will compare a small quantum experiment against this classical reference. Use the
**same original-input validator** for every quantum sample; keep invalid raw samples
separate from any repaired results. Hardware access and budgets must be checked before
submitting jobs. H1 introduces no quantum-service cost.